# Custom Hamiltonian and guide classes

This notebook shows how to define a custom Hamiltonian and a custom guiding wavefunction, using the
examples from the appendix of the paper. The Hamiltonian is an $n$-body model in which every term acts
on a whole cluster of sites,

$$H = -J_z \sum_{c\,\in\, \mathcal{C}} \prod_{i\in c} \sigma^z_i \;-\; J_x \sum_{c\,\in\, \mathcal{C}} \prod_{i\in c} \sigma^x_i \;-\; h \sum_i \sigma^z_i\,,$$

where $\mathcal{C}$ is a collection of $n$-site clusters, written in Pauli operators. The $\sigma^z$ terms are diagonal, and the $\sigma^x$ part flips every site of a cluster at once, so each cluster contributes
exactly one off-diagonal transition. The model is sign-problem free for $J_x \ge 0$.

In [1]:
import jax
import jax.numpy as jnp
import numpy as np

from pygfmc import run, evaluate_observables
from pygfmc.walkers import prepare_walkers_random
from pygfmc.hamiltonian import exact_diagonalization

print(f'JAX version : {jax.__version__}')
print(f'Backend     : {jax.default_backend()}')

JAX version : 0.9.2
Backend     : cpu


## Custom Hamiltonian class

We use four-site
clusters ($n = 4$) of consecutive sites on a periodic chain of $L = 12$ sites, with $J_z = 1$,
$J_x = 0.5$ and $h = 0.5$.

A Hamiltonian is any object with three required members and one optional one:

* `M_total`: the number of terms, one diagonal term plus one per possible off-diagonal transition (here one per cluster);
* `get_rates(config, return_configs=False)`: returns the matrix elements of all terms acting on `config`; the first entry belongs to the diagonal; with `return_configs=True`, it also returns the successor configurations;
* `generate_state(config, idx)`: the configuration generated by term `idx`;
* `diagonal_weight(config)` (optional): returns the diagonal value.

The class is registered as a JAX pytree, so that JAX can trace through it inside the compiled loops.

In [2]:
class NBody:
    def __init__(self, clusters, J_z, J_x, h=0.0):
        self.clusters = jnp.asarray(clusters)          # (n_terms, n)
        self.J_z, self.J_x, self.h = J_z, J_x, h
        self.M_total  = 1 + self.clusters.shape[0]

    def get_rates(self, config, return_configs=False):
        diag  = self.diagonal_weight(config)
        off   = jnp.full(self.clusters.shape[0], -self.J_x)
        rates = jnp.concatenate([jnp.array([diag]), off])
        if not return_configs:
            return rates, None
        flips = jax.vmap(lambda c: config.at[c].multiply(-1))(self.clusters)
        return rates, jnp.concatenate([config[None, :], flips])

    def generate_state(self, config, idx):
        flipped = config.at[self.clusters[idx - 1]].multiply(-1)
        return jnp.where(idx == 0, config, flipped)

    def diagonal_weight(self, config):
        return (-self.J_z * jnp.sum(jnp.prod(config[self.clusters], axis=1))
                - self.h * jnp.sum(config))

jax.tree_util.register_pytree_node(
    NBody,
    lambda h: ((h.clusters,), (h.J_z, h.J_x, h.h)),
    lambda aux, children: NBody(children[0], *aux))

## Exact reference and GFMC run

`exact_diagonalization` accepts any class that follows the protocol, given the local states of a site
($\pm 1$ for spins). We start from random configurations and
run the default continuous-time propagation with `step_size = 0.25` and the projection windows
$p = 2, 4, 8$.

In [3]:
L        = 12
clusters = [[(i + k) % L for k in range(4)] for i in range(L)]   # four consecutive sites
H        = NBody(clusters, J_z=1.0, J_x=0.5, h=0.5)

E_ed = exact_diagonalization(H, local_hilbert_space=(-1, 1), N=L, number_of_states=4)[0]
E0   = float(E_ed[0])
print(f'M_total = {H.M_total}; lowest ED energies: {np.round(np.asarray(E_ed), 6)}')

N_WALKERS, N_STEPS, P_ARRAY = 1024, 3000, [2, 4, 8]

walkers = prepare_walkers_random(N_WALKERS, L, jax.random.PRNGKey(0))
_, energy, _, stats, raw = run(
    walkers, N_STEPS, H, step_size=0.25, guide=None, p_array=P_ARRAY,
    rng_key=jax.random.PRNGKey(1), warmup_steps=500, verbosity=8, number_of_outputs=20
)
E, _, E_err, _ = evaluate_observables(raw, n_blocks=20)
for p, n_eff in zip(P_ARRAY, stats['effective_sample_size']):
    print(f'p = {p}:  E = {E[p]:.4f} +- {E_err[p]:.4f}   (ED {E0:.4f})   n_eff = {n_eff:.0f}')


M_total = 13; lowest ED energies: [-18.25255  -13.352273 -13.352273 -12.728587]


[gfmc] - [run_warmup 1/4] - step=0.03125, 125 steps @ 23.7 it/s — avg_log_weight = 16.589020


[gfmc] - [run_warmup 2/4] - step=0.0625, 125 steps @ 36.7 it/s — avg_log_weight = 18.271164


[gfmc] - [run_warmup 3/4] - step=0.125, 125 steps @ 36.7 it/s — avg_log_weight = 18.193138


[gfmc] - [run_warmup 4/4] - step=0.25, 125 steps @ 34.2 it/s — avg_log_weight = 18.226469


...

[gfmc] - [run_gfmc] - step: 2550/3000, 261.7 it/s, N_config = 84/1024 (8.2%) Energy[p]: {2: -18.261959562301897, 4: -18.262108921219106, 8: -18.260094653949402}


[gfmc] - [run_gfmc] - step: 2700/3000, 260.7 it/s, N_config = 106/1024 (10.4%) Energy[p]: {2: -18.261722972853526, 4: -18.261810536178377, 8: -18.25997956568366}


[gfmc] - [run_gfmc] - step: 2850/3000, 247.0 it/s, N_config = 97/1024 (9.5%) Energy[p]: {2: -18.261718152203542, 4: -18.261890518288276, 8: -18.26022121137961}


[gfmc] - [run_gfmc] - step: 3000/3000, 223.7 it/s, N_config = 96/1024 (9.4%) Energy[p]: {2: -18.26128431512333, 4: -18.261712684914983, 8: -18.260123075529897}


[estimators] - [statistics] - 10 branching steps
[estimators] - [statistics] - N_walkers: 1024
[estimators] - [statistics] - step_size: 0.25
[estimators] - [statistics] - lam: not used
[estimators] - [statistics] - avg_log_weight: 18.226469
[estimators] - [statistics] - unique (pre): 95.7 ± 5.4  (9.3%)
[estimators] - [statistics] - unique (post): 95.6 ± 5.4  (9.3%)
[estimators] - [statistics] - H_diag: min=-18.0000, max=8.0000  (observed max over the 10 health steps)
[estimators] - [statistics] - escape rate: min=6.0000, max=6.0000
[estimators] - [statistics] - hops per branching step (Gamma*step_size): mean=1.500, max=1.500
[estimators] - [statistics] - escape rate Gamma: mean=6.000, min=6.000, max=6.000
[estimators] - [statistics] - hold time (a draw of 1/Gamma, not hops per step): mean=0.1657, max=1.6164
[estimators] - [statistics] - changed (per-index): 66.4% avg over 10 steps
[estimators] - [statistics] - overlap (pop):       59.2% avg over 10 steps
[estimators] - [statistics] - o

## Custom guiding wavefunction

`MagnetizationGuide` from the paper implements

$$\log\psi_T(x) = \beta \sum_i x_i\,.$$

A guide needs at least one of `log_diff_from_operator(config, op, cache=None)`,
`log_diff_from_configs(config, new_configs, cache=None)` or `log_psi(config, cache=None)`. The variational parameter $\beta$ is the only child of the pytree.

In [4]:
class MagnetizationGuide:
    def __init__(self, beta):
        self.beta = jnp.asarray(beta)

    def log_psi(self, config, cache=None):
        # config is (N,) or (n, N)
        return self.beta * jnp.sum(config, axis=-1)

    def log_diff_from_configs(self, config, new_configs, cache=None):
        return self.log_psi(new_configs) - self.log_psi(config)

    def log_diff_from_operator(self, config, op, cache=None):
        # flipping the sites of cluster k changes the magnetization
        # by -2 sum_{i in cluster k} x_i; entry 0 is the diagonal
        dm = -2.0 * jnp.sum(config[op.clusters], axis=1)
        return self.beta * jnp.concatenate([jnp.zeros(1, dtype=dm.dtype), dm])

jax.tree_util.register_pytree_node(
    MagnetizationGuide,
    lambda g: ((g.beta,), None),
    lambda aux, children: MagnetizationGuide(*children),
)

All three routines must agree. We compare the fast `log_diff_from_operator` with `log_psi` evaluated on
the successor configurations that `get_rates` returns:

In [5]:
guide = MagnetizationGuide(beta=0.2)

config = prepare_walkers_random(1, L, jax.random.PRNGKey(5)).config[0]
_, successors = H.get_rates(config, return_configs=True)
fast = guide.log_diff_from_operator(config, H)
slow = guide.log_psi(successors) - guide.log_psi(config)
print('max deviation:', float(jnp.max(jnp.abs(fast - slow))))

max deviation: 0.0


## Training the guide with NetKet

We train the new guide with NetKet:

In [6]:
from pygfmc.wavefunction import train_with_netket
import os
os.environ["NETKET_NO_TIPS"] = "1"   # silence NetKet's tips
import netket as nk          # after pygfmc, so that everything stays in float32

sampler = nk.sampler.MetropolisLocal(nk.hilbert.Spin(0.5, N=L))
trained, history = train_with_netket(H, MagnetizationGuide(beta=0.0), sampler=sampler,
                                     return_history=True)

print(f'trained beta = {float(trained.beta):.3f}')
print(f'variational energy: {history[0]:.4f} (beta = 0) -> {np.mean(history[-10:]):.4f} (trained);  ED {E0:.4f}')

trained beta = 2.052
variational energy: -6.1621 (beta = 0) -> -17.9719 (trained);  ED -18.2526
